In [1]:
import pandas as pd
a_attr = pd.read_json("../data/Anno/a_attr.json", lines=True)
c_attr = pd.read_json("../data/Anno/c_attr.json", lines=True)
p_attr = pd.read_json("../data/Anno/p_attr.json", lines=True)
s_attr = pd.read_json("../data/Anno/s_attr.json", lines=True)

In [2]:
import json

with open("../data/croissant/metadata.json") as f:
    meta = json.load(f)

attr_mapping = {}
for field in meta["schema"]["fields"]:
    if field["type"] == "integer":
        desc = field["description"]
        parts = [p.strip() for p in desc.split(",")]
        mapping = {}
        for p in parts:
            val, label = p.split("=")
            mapping[int(val.strip())] = label.strip()
        attr_mapping[field["name"]] = mapping


print(attr_mapping)
print(a_attr["gender"].map(mapping))

{'gender': {1: 'male', 2: 'female'}, 'Asian': {1: 'yes', 2: 'no'}, 'Black': {1: 'yes', 2: 'no'}, 'Others': {1: 'yes', 2: 'no'}, 'teenager': {1: 'yes', 2: 'no'}, 'middle': {1: 'yes', 2: 'no'}, 'elderly': {1: 'yes', 2: 'no'}, 'wrinkles': {1: 'yes', 2: 'no'}, 'sideburns': {1: 'yes', 2: 'no'}, 'moustache/goatee': {1: 'yes', 2: 'no'}, 'beard': {1: 'yes', 2: 'no'}, 'clear_glasses': {1: 'yes', 2: 'no'}, 'sunglasses': {1: 'yes', 2: 'no'}, 'glasses': {1: 'yes', 2: 'no'}, 'open_eyes': {1: 'yes', 2: 'no'}, 'eyebrows': {1: 'yes', 2: 'no'}, 'open_mouth': {1: 'yes', 2: 'no'}, 'show_teeth': {1: 'yes', 2: 'no'}, 'smile_with_closed_lips': {1: 'yes', 2: 'no'}, 'smile_with_open_lips': {1: 'yes', 2: 'no'}, 'show_ears': {1: 'yes', 2: 'no'}, 'earring': {1: 'yes', 2: 'no'}, 'pointed_nose': {1: 'yes', 2: 'no'}, 'round_nose': {1: 'yes', 2: 'no'}, 'pointed_chin': {1: 'yes', 2: 'no'}, 'round_chin': {1: 'yes', 2: 'no'}, 'bald': {1: 'yes', 2: 'no'}, 'sparse_hair': {1: 'yes', 2: 'no'}, 'short_hair': {1: 'yes', 2: '

In [3]:
from PIL import Image

img = Image.open('../data/Img/Art/a00001.jpg')
img.show()

In [4]:
# Construction d'une variable long_hair == 1 et (smile_with_closed_lips == 1 or smile_with_open_lips == 1) dans p_attr
p_attr["label"] = (p_attr["long_hair"] == 1) & ((p_attr["smile_with_closed_lips"] == 1) | (p_attr["smile_with_open_lips"] == 1))

In [9]:
# Tableaux croisés en pourcentages de p_attr["label"] avec les autres variables de p_attr (sauf ID)
for col in p_attr.columns:
    if col != "label" and col != "ID":
        print(f"Tableau croisé de label avec {col}:")
        print(pd.crosstab(p_attr["label"], p_attr[col], normalize="index"))

Tableau croisé de label avec gender:
gender         1         2
label                     
False   0.585865  0.414135
True    0.023063  0.976937
Tableau croisé de label avec Asian:
Asian         1         2
label                    
False  0.862683  0.137317
True   0.873265  0.126735
Tableau croisé de label avec Black:
Black         1         2
label                    
False  0.076182  0.923818
True   0.091245  0.908755
Tableau croisé de label avec Others:
Others         1         2
label                     
False   0.061135  0.938865
True    0.035490  0.964510
Tableau croisé de label avec teenager:
teenager         1         2
label                       
False     0.482770  0.517230
True      0.653829  0.346171
Tableau croisé de label avec middle:
middle         1         2
label                     
False   0.456901  0.543099
True    0.340461  0.659539
Tableau croisé de label avec elderly:
elderly         1         2
label                      
False    0.060328  0.939672
True    

In [ ]:
from sklearn.model_selection import train_test_split
X = p_attr.drop(columns=["label"])
y = p_attr["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os   


class FaceDataset(Dataset):
    def __init__(self, img_dir, data, label, transform=None):
        self.img_dir = img_dir
        self.data = data
        self.label = label
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.iloc[idx]["ID"]
        label = self.label.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{img_id.strip()}.jpg")
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Création du dataset et dataloader
dataset = FaceDataset(img_dir="../data/Img/Photo", data=X_train, label=y_train, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = models.efficientnet_b0(weights="EfficientNet_B0_Weights.DEFAULT")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)   
num_epochs = 5


for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    i = 0
    TP = 0
    FP = 0
    TN = 0
    FN = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.float().to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0) 
        predicted = (outputs > 0).float()
        TP += ((predicted == 1) & (labels == 1)).sum().item()
        FP += ((predicted == 1) & (labels == 0)).sum().item()
        TN += ((predicted == 0) & (labels == 0)).sum().item()
        FN += ((predicted == 0) & (labels == 1)).sum().item()
        if i % 50 == 0:
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")
            print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
        i += 1

    F1_score = 2 * TP / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 0
    print(f"Epoch {epoch+1}, F1 Score: {F1_score:.4f}")
    epoch_loss = running_loss / len(dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")





Epoch 1, Batch 0, Loss: 0.7227
TP: 1, FP: 16, TN: 11, FN: 4
Epoch 1, Batch 50, Loss: 0.4894
TP: 256, FP: 190, TN: 996, FN: 190
Epoch 1, Batch 100, Loss: 0.1508
TP: 579, FP: 282, TN: 2061, FN: 310
Epoch 1, Batch 150, Loss: 0.2501
TP: 941, FP: 373, TN: 3094, FN: 424
Epoch 1, Batch 200, Loss: 0.2724
TP: 1306, FP: 458, TN: 4143, FN: 525
Epoch 1, Batch 250, Loss: 0.3190
TP: 1684, FP: 546, TN: 5167, FN: 635
Epoch 1, Batch 300, Loss: 0.1738
TP: 2053, FP: 617, TN: 6226, FN: 736
Epoch 1, Batch 350, Loss: 0.2481
TP: 2427, FP: 695, TN: 7286, FN: 824
Epoch 1, Batch 400, Loss: 0.2514
TP: 2814, FP: 767, TN: 8330, FN: 921
Epoch 1, Batch 450, Loss: 0.2491
TP: 3222, FP: 845, TN: 9365, FN: 1000
Epoch 1, Batch 500, Loss: 0.2138
TP: 3613, FP: 920, TN: 10400, FN: 1099
Epoch 1, Batch 550, Loss: 0.3855
TP: 3995, FP: 994, TN: 11470, FN: 1173
Epoch 1, Batch 600, Loss: 0.3018
TP: 4416, FP: 1064, TN: 12490, FN: 1262
Epoch 1, Batch 650, Loss: 0.1124
TP: 4819, FP: 1159, TN: 13519, FN: 1335
Epoch 1, Batch 700, Loss

In [23]:
# calcul de la matrice de confusion et du F1 score sur le jeu de test
model.eval()
test_dataset = FaceDataset(img_dir="../data/Img/Photo", data=X_test, label=y_test, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
TP = 0
FP = 0
TN = 0
FN = 0
with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.float().to(device)
        outputs = model(images).squeeze()
        predicted = (outputs > 0).float()
        TP += ((predicted == 1) & (labels == 1)).sum().item()
        FP += ((predicted == 1) & (labels == 0)).sum().item()
        TN += ((predicted == 0) & (labels == 0)).sum().item()
        FN += ((predicted == 0) & (labels == 1)).sum().item()
F1_score = 2 * TP / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 0
print(f"Test F1 Score: {F1_score:.4f}")


Test F1 Score: 0.8379


In [24]:
# Sauvegarde du modèle
torch.save(model.state_dict(), "../models/face_effnet_b0_epoch5_f1_0.84.pth")
